# TILDE (2021)
---
[[paper]](https://arxiv.org/abs/2108.13111)<br>
TILDE = Term Independent Likelihood moDEl

__TILDE__ — это метод Learned Sparse Retrieval для первого этапа поиска (First Stage Retrieval). Он использует BERT для предсказания вероятности появления каждого слова из всего словаря (vocabulary) в потенциальном запросе к данному документу. Это позволяет строить очень эффективные разреженные индексы, которые работают со скоростью классического BM25, но обладают семантическим пониманием уровня трансформерных моделей.

__Постановка задачи__<br>
Дана коллекция документов $D$. По входящему текстовому запросу $Q$ необходимо максимально быстро вернуть топ-K наиболее релевантных документов. Основное ограничение — поиск должен быть масштабируемым на миллионы документов, что обычно подразумевает использование инвертированного индекса.

__Мотивация__<br>
Существующие на тот момент модели либо были очень медленными (Cross-Encoders), либо требовали сложной инфраструктуры для векторного поиска (Dense Retrieval), либо страдали от проблемы Vocabulary Mismatch (BM25). Авторы хотели создать модель, которая делает расширение документа (Document Expansion) так же качественно, как нейросети, но без необходимости генерации текста (как в Doc2query) и с сохранением структуры инвертированного индекса.

__Существующие подходы__<br>
На момент появления TILDE (2021) основными альтернативами были:
- BM25 (1990-е): использует точное совпадение слов. Быстро, но не понимает синонимы.
- Doc2query (2019): использует T5 для генерации новых слов/вопросов к документу. Главный минус — очень медленная индексация из-за авторегрессионной генерации (нужно генерировать по 40-80 токенов на каждый пассаж).
- DeepCT (2019): перевзвешивает только те слова, которые уже есть в документе. Не решает проблему отсутствия нужных слов (Vocabulary Mismatch).
- SPLADE (2021): использует MLM head и разреженную регуляризацию. Очень эффективно, но архитектурно сложнее и на тот момент требовало тщательного подбора коэффициентов регуляризации для контроля разреженности.

__Идея__<br>
Основная идея TILDE заключается в предположении о независимости термов (Term Independence). Вместо того чтобы моделировать вероятность запроса целиком, модель предсказывает вероятность появления каждого слова из словаря $V$ в запросе $Q$ при условии документа $D$ независимо друг от друга. Это превращает задачу Document Expansion в задачу многоклассовой классификации (или регрессии) над всем словарем токенов BERT.

__Архитектура__<br>
TILDE базируется на архитектуре BERT (обычно `distilbert-base-uncased` для скорости):
1.  Input: Текст документа, дополненный специальным токеном `[CLS]`.
2.  Encoder: BERT обрабатывает документ и возвращает контекстуализированные векторы.
3.  Projection Head: Используется вектор `[CLS]`-токена с последнего слоя. Он проходит через полносвязный слой, который проецирует его в пространство размерности $|V|$ (размер словаря BERT, около 30 522 токенов).
4.  Output: Вектор логитов $L$, где каждый элемент $L_i$ соответствует вероятности того, что i-й токен из словаря будет присутствовать в релевантном запросе.

__Алгоритм обучения__<br>
Модель обучается на парах (запрос, документ) из датасета MS MARCO:
1.  Для каждого документа в обучающей выборке мы знаем "золотой" запрос.
2.  Модель предсказывает веса для всех 30k токенов словаря.
3.  Loss function: Используется Binary Cross Entropy (BCE). Для слов, которые реально присутствуют в запросе, целевое значение — 1, для остальных — 0. 
4.  В отличие от стандартного языкового моделирования, здесь нет маскирования. Модель учится по всему документу предсказывать именно слова запроса.

__Алгоритм инференса__<br>
Процесс разделен на offline индексацию и online поиск:

1.  Offline Indexing:
    - Каждый документ пропускается через TILDE.
    - Из выходного вектора выбираются топ-K токенов с наибольшими весами (например, топ-200 слов).
    - Эти веса квантуются (превращаются в целые числа) для экономии места.
    - В инвертированный индекс записываются эти выбранные токены с их новыми весами. Это по сути "обогащенный" документ.
    
2.  Online Retrieval:
    - Запрос не пропускается через нейросеть (это называется Query-Free Inference в контексте TILDE). 
    - Мы просто берем токены запроса и ищем их в нашем инвертированном индексе, суммируя веса, предсказанные моделью на этапе индексации.
    - Это дает колоссальный прирост скорости, так как на этапе поиска нет вообще никакой работы GPU.

__Результаты__<br>
Сравнение проводилось на MS MARCO Passage Ranking:
- Эффективность: TILDE достигает MRR@10 около 0.33-0.34, что на 15-18пп выше, чем у классического BM25 (0.18).
- Скорость: TILDE выполняет запрос за 10-20 мс на CPU, что идентично скорости BM25, в то время как Cross-Encoders требуют сотни миллисекунд и GPU.
- Скорость индексации: TILDE в 10-50 раз быстрее, чем Doc2query, так как делает один прямой проход (forward pass) вместо долгой генерации текста.
- В расширенной версии (TILDEv2) авторы добавили обработку запроса (Query Encoder), что позволило еще больше сократить разрыв с Dense Retrieval моделями, сохранив при этом разреженную структуру индекса.

## 📝 Критический анализ

```markdown
# TILDE (2021)
---
[[paper]](https://arxiv.org/abs/2108.13111)<br>
TILDE = Term Independent Likelihood moDEl

**TILDE** — метод Learned Sparse Retrieval для первого этапа поиска. Использует BERT для предсказания вероятности появления каждого слова из словаря в запросе к документу, создавая разреженные индексы с семантическим пониманием уровня трансформеров.

__Постановка задачи__<br>
По запросу $Q$ вернуть топ-K релевантных документов из коллекции $D$. Поиск должен быть масштабируемым на миллионы документов, используя инвертированный индекс.

__Мотивация__<br>
Существующие модели были либо медленными, либо требовали сложной инфраструктуры, либо страдали от Vocabulary Mismatch. TILDE стремится к качественному Document Expansion без генерации текста и с сохранением структуры инвертированного индекса.

__Существующие подходы__<br>
- BM25: быстро, но не понимает синонимы.
- Doc2query: медленная индексация из-за генерации.
- DeepCT: не решает Vocabulary Mismatch.
- SPLADE: эффективно, но сложнее в настройке.

__Идея__<br>
Предполагается независимость термов. Модель предсказывает вероятность каждого слова из словаря $V$ в запросе $Q$ при условии документа $D$, превращая Document Expansion в многоклассовую классификацию.

__Архитектура__<br>
<img src="img/img.png" width=500>
1. Input: Документ с токеном `[CLS]`.
2. Encoder: BERT возвращает контекстуализированные векторы.
3. Projection Head: Вектор `[CLS]` проецируется в пространство размерности $|V|$.
4. Output: Вектор логитов $L$, где $L_i$ — вероятность i-го токена в запросе.

__Алгоритм обучения__<br>
Обучение на парах (запрос, документ) из MS MARCO:
1. Предсказание весов для всех токенов словаря.
2. Loss: Binary Cross Entropy. Целевое значение — 1 для присутствующих в запросе слов, 0 — для остальных.

__Алгоритм инференса__<br>
1. Offline Indexing:
   - Пропуск документа через TILDE.
   - Выбор топ-K токенов с наибольшими весами.
   - Квантуются веса и записываются в инвертированный индекс.

2. Online Retrieval:
   - Запрос не проходит через нейросеть.
   - Поиск токенов запроса в индексе, суммирование весов.

__Результаты__<br>
- Эффективность: MRR@10 около 0.33-0.34, на 15-18пп выше BM25.
- Скорость: Запрос за 10-20 мс на CPU, аналогично BM25.
- Индексация: В 10-50 раз быстрее Doc2query.
- TILDEv2 добавляет Query Encoder, сокращая разрыв с Dense Retrieval моделями.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel
import torch.nn as nn
import torch.nn.functional as F

# Определяем модель TILDE
class TILDE(nn.Module):
    def __init__(self, vocab_size):
        super(TILDE, self).__init__()
        # Используем DistilBERT в качестве энкодера
        self.encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        # Полносвязный слой для проекции в пространство размерности словаря
        self.projection = nn.Linear(self.encoder.config.hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask):
        # Получаем вектор [CLS] из DistilBERT
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # Вектор [CLS]
        # Проецируем в пространство размерности словаря
        logits = self.projection(cls_output)
        return logits

# Инициализация токенизатора и модели
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
vocab_size = len(tokenizer)  # Размер словаря BERT
model = TILDE(vocab_size)

# Пример документа
document = "The quick brown fox jumps over the lazy dog."

# Токенизация документа
inputs = tokenizer(document, return_tensors='pt', truncation=True, padding=True)

# Прогоняем документ через модель
logits = model(inputs['input_ids'], inputs['attention_mask'])

# Применяем сигмоиду для получения вероятностей
probabilities = torch.sigmoid(logits)

# Выбираем топ-K токенов с наибольшими вероятностями
top_k = 10
top_k_probs, top_k_indices = torch.topk(probabilities, top_k, dim=1)

# Выводим топ-K токенов и их вероятности
top_k_tokens = [tokenizer.decode([idx]) for idx in top_k_indices[0]]
print("Top-K tokens:", top_k_tokens)
print("Probabilities:", top_k_probs[0].tolist())

# Пример индексации
# В реальной системе мы бы сохранили top_k_tokens и их вероятности в инвертированный индекс

# Пример поиска
# Для поиска мы бы использовали токены запроса и искали их в инвертированном индексе
query = "fast fox"
query_tokens = tokenizer.tokenize(query)

# Псевдокод для поиска
# results = search_inverted_index(query_tokens)
# print("Retrieved documents:", results)
```

### Объяснение ключевых моментов:

1. **Архитектура**: Мы используем `DistilBERT` для получения контекстуализированного вектора `[CLS]`, который затем проецируется в пространство размерности словаря BERT с помощью полносвязного слоя.

2. **Предсказание вероятностей**: Модель предсказывает вероятности для каждого токена из словаря, используя сигмоидную функцию активации.

3. **Индексация**: На этапе индексации мы выбираем топ-K токенов с наибольшими вероятностями и сохраняем их в инвертированном индексе. Это позволяет эффективно обогащать документы.

4. **Поиск**: На этапе поиска мы используем токены запроса для поиска в инвертированном индексе, что делает процесс очень быстрым, так как не требует прогонки запроса через нейросеть.

Этот пример демонстрирует, как TILDE использует BERT для предсказания вероятностей появления токенов в запросе, сохраняя при этом эффективность разреженного поиска.